# SQL 정리 — 실기 대비

> SQL은 **2~3문항(10~15점)**. 문법만 정확히 외우면 거의 다 맞을 수 있는 구간이다.
> 실기는 손으로 쓰기 때문에 **세미콜론, 괄호, 콤마**까지 정확히 써야 한다.

---

## 1. SQL 분류

| 분류 | 명령어 | 설명 |
|---|---|---|
| **DDL** (정의어) | CREATE, ALTER, DROP, TRUNCATE | 구조 정의 |
| **DML** (조작어) | SELECT, INSERT, UPDATE, DELETE | 데이터 조작 |
| **DCL** (제어어) | GRANT, REVOKE, COMMIT, ROLLBACK, SAVEPOINT | 권한 / 트랜잭션 |

> TCL(COMMIT/ROLLBACK/SAVEPOINT)을 따로 분류하기도 한다. 문제에 따라 DCL에 포함시켜 답한다.

---

## 2. DDL

### CREATE TABLE

```sql
CREATE TABLE 학생 (
    학번   CHAR(8)      PRIMARY KEY,
    이름   VARCHAR(20)  NOT NULL,
    학과   VARCHAR(30)  DEFAULT '미정',
    성적   NUMBER(3)    CHECK (성적 >= 0 AND 성적 <= 100),
    지도교수 CHAR(8),
    FOREIGN KEY (지도교수) REFERENCES 교수(교수번호)
        ON DELETE CASCADE
);
```

**제약조건 6가지**
- `PRIMARY KEY` : 기본키 (유일 + NULL 불가)
- `FOREIGN KEY ~ REFERENCES` : 외래키
- `UNIQUE` : 중복 불가 (NULL 허용)
- `NOT NULL` : NULL 불가
- `CHECK` : 값의 범위 제한
- `DEFAULT` : 기본값

**참조 옵션**: `ON DELETE CASCADE` (연쇄 삭제) / `SET NULL` / `SET DEFAULT` / `RESTRICT` / `NO ACTION`

### ALTER TABLE

```sql
ALTER TABLE 학생 ADD 전화번호 VARCHAR(20);            -- 속성 추가
ALTER TABLE 학생 MODIFY 이름 VARCHAR(30) NOT NULL;     -- 속성 변경
ALTER TABLE 학생 DROP COLUMN 전화번호;                 -- 속성 삭제
```

### DROP / TRUNCATE

```sql
DROP TABLE 학생 CASCADE;    -- 참조하는 것까지 함께 삭제
DROP TABLE 학생 RESTRICT;   -- 참조 중이면 삭제 취소
TRUNCATE TABLE 학생;        -- 구조는 남기고 데이터만 전부 삭제
```

**DELETE vs TRUNCATE vs DROP**

| | DELETE | TRUNCATE | DROP |
|---|---|---|---|
| 분류 | DML | DDL | DDL |
| 삭제 대상 | 행(조건 가능) | 전체 행 | 테이블 자체 |
| ROLLBACK | 가능 | 불가 | 불가 |

---

## 3. DML

### INSERT

```sql
INSERT INTO 학생(학번, 이름, 학과) VALUES ('20250001', '홍길동', '컴퓨터공학');
INSERT INTO 학생 VALUES ('20250002', '김철수', '전자공학', 90, NULL);  -- 전체 컬럼
```

### UPDATE

```sql
UPDATE 학생
SET 학과 = '소프트웨어학과', 성적 = 95
WHERE 학번 = '20250001';
```

### DELETE

```sql
DELETE FROM 학생 WHERE 성적 < 60;
DELETE FROM 학생;               -- 전체 행 삭제 (테이블은 남음)
```

### SELECT 기본형

```sql
SELECT [DISTINCT] 속성리스트
FROM 테이블
WHERE 조건
GROUP BY 속성
HAVING 그룹조건
ORDER BY 속성 [ASC | DESC];
```

**실행 순서 (암기 필수)**
```
FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY
```

---

## 4. SELECT 심화

### WHERE 조건절

```sql
WHERE 성적 >= 80
WHERE 성적 BETWEEN 80 AND 90          -- 80 이상 90 이하 (양끝 포함)
WHERE 학과 IN ('컴공', '전자')
WHERE 이름 LIKE '김%'                 -- 김으로 시작
WHERE 이름 LIKE '_길동'               -- 한 글자 + 길동
WHERE 지도교수 IS NULL                -- NULL 비교는 = 가 아니라 IS
WHERE NOT 학과 = '컴공'
```

**LIKE 와일드카드: `%` = 0글자 이상, `_` = 정확히 1글자**

### 집계 함수

```sql
SELECT COUNT(*), COUNT(성적), SUM(성적), AVG(성적), MAX(성적), MIN(성적)
FROM 학생;
```
- `COUNT(*)` : NULL 포함 전체 행 수
- `COUNT(컬럼)` : **NULL 제외** 행 수
- `AVG`, `SUM` 등도 **NULL은 계산에서 제외**

### GROUP BY + HAVING

```sql
SELECT 학과, COUNT(*) AS 인원수, AVG(성적) AS 평균
FROM 학생
WHERE 성적 >= 60           -- 행에 대한 조건
GROUP BY 학과
HAVING COUNT(*) >= 3       -- 그룹에 대한 조건
ORDER BY 평균 DESC;
```

**WHERE vs HAVING**: WHERE는 그룹화 **전** 개별 행 조건, HAVING은 그룹화 **후** 그룹 조건. 집계 함수는 HAVING에만 쓸 수 있다.

### ORDER BY

```sql
ORDER BY 성적 DESC, 이름 ASC;    -- 성적 내림차순, 같으면 이름 오름차순
ORDER BY 2 DESC;                 -- SELECT 목록의 두 번째 컬럼 기준
```

---

## 5. JOIN

### INNER JOIN (내부 조인)

```sql
-- ANSI 표준
SELECT s.이름, d.학과명
FROM 학생 s INNER JOIN 학과 d
ON s.학과코드 = d.학과코드;

-- WHERE 방식 (동등 조인)
SELECT s.이름, d.학과명
FROM 학생 s, 학과 d
WHERE s.학과코드 = d.학과코드;
```

### OUTER JOIN (외부 조인)

```sql
SELECT s.이름, d.학과명
FROM 학생 s LEFT OUTER JOIN 학과 d
ON s.학과코드 = d.학과코드;
```
- **LEFT OUTER**: 왼쪽 테이블은 전부 + 오른쪽은 매칭되는 것 (없으면 NULL)
- **RIGHT OUTER**: 반대
- **FULL OUTER**: 양쪽 전부

### 기타 조인

```sql
SELECT * FROM 학생 CROSS JOIN 학과;         -- 카티션 프로덕트 (모든 조합)
SELECT * FROM 학생 NATURAL JOIN 학과;       -- 같은 이름 컬럼으로 자동 조인
SELECT a.이름, b.이름 FROM 사원 a, 사원 b   -- SELF JOIN (같은 테이블)
WHERE a.상사번호 = b.사번;
```

---

## 6. 서브쿼리

### 단일 행 서브쿼리

```sql
SELECT 이름 FROM 학생
WHERE 성적 > (SELECT AVG(성적) FROM 학생);
```

### 다중 행 서브쿼리

```sql
SELECT 이름 FROM 학생
WHERE 학과코드 IN (SELECT 학과코드 FROM 학과 WHERE 단과대 = '공대');

SELECT 이름 FROM 학생
WHERE 성적 > ALL (SELECT 성적 FROM 학생 WHERE 학과 = '컴공');   -- 모두보다 큰

SELECT 이름 FROM 학생
WHERE 성적 > ANY (SELECT 성적 FROM 학생 WHERE 학과 = '컴공');   -- 하나라도보다 큰
```

### EXISTS

```sql
SELECT 이름 FROM 학생 s
WHERE EXISTS (SELECT 1 FROM 수강 c WHERE c.학번 = s.학번);
```

---

## 7. 집합 연산

```sql
SELECT 이름 FROM A UNION     SELECT 이름 FROM B;   -- 합집합 (중복 제거)
SELECT 이름 FROM A UNION ALL SELECT 이름 FROM B;   -- 합집합 (중복 유지)
SELECT 이름 FROM A INTERSECT SELECT 이름 FROM B;   -- 교집합
SELECT 이름 FROM A EXCEPT    SELECT 이름 FROM B;   -- 차집합 (Oracle은 MINUS)
```

---

## 8. DCL

### GRANT / REVOKE

```sql
GRANT SELECT, INSERT ON 학생 TO 홍길동;
GRANT ALL ON 학생 TO 홍길동 WITH GRANT OPTION;   -- 권한 재부여 가능

REVOKE SELECT ON 학생 FROM 홍길동;
REVOKE SELECT ON 학생 FROM 홍길동 CASCADE;       -- 연쇄 회수
```

**형식 암기: `GRANT 권한 ON 대상 TO 사용자` / `REVOKE 권한 ON 대상 FROM 사용자`**
→ TO와 FROM을 헷갈리는 실수가 매우 많다.

### 트랜잭션 제어

```sql
COMMIT;                  -- 확정
ROLLBACK;                -- 취소
SAVEPOINT sp1;           -- 저장점 설정
ROLLBACK TO sp1;         -- 저장점까지 취소
```

---

## 9. VIEW / INDEX

### VIEW

```sql
CREATE VIEW 우수학생 AS
SELECT 학번, 이름, 성적 FROM 학생 WHERE 성적 >= 90;

DROP VIEW 우수학생;
```
- 뷰는 **가상 테이블** (실제 데이터 저장 X)
- 뷰의 정의는 **변경 불가** → 수정하려면 DROP 후 재생성
- 뷰를 기반으로 또 다른 뷰 생성 가능

### INDEX

```sql
CREATE INDEX idx_이름 ON 학생(이름);
CREATE UNIQUE INDEX idx_학번 ON 학생(학번);
DROP INDEX idx_이름;
```
- 검색 속도 향상, 그러나 **INSERT/UPDATE/DELETE 성능은 저하**

---

## 10. 트리거 / 프로시저

### 트리거

```sql
CREATE TRIGGER 학생_로그
AFTER INSERT ON 학생
FOR EACH ROW
BEGIN
    INSERT INTO 로그 VALUES (:NEW.학번, SYSDATE);
END;
```
- 시점: `BEFORE` / `AFTER`
- 이벤트: `INSERT` / `UPDATE` / `DELETE`
- **트리거 내부에서는 COMMIT / ROLLBACK 사용 불가**

### 프로시저

```sql
CREATE PROCEDURE 성적갱신(IN p_학번 CHAR(8), IN p_성적 INT)
BEGIN
    UPDATE 학생 SET 성적 = p_성적 WHERE 학번 = p_학번;
END;
```
- 매개변수 유형: `IN` (입력) / `OUT` (출력) / `INOUT`
- 프로시저 실행: `CALL 성적갱신('20250001', 95);`
- **프로시저 vs 함수**: 함수는 반드시 값을 반환(RETURN), 프로시저는 반환값이 없어도 됨

---

## 11. 자주 나오는 함수

```sql
UPPER('abc')                  -- 'ABC'
LOWER('ABC')                  -- 'abc'
SUBSTR('abcdef', 2, 3)        -- 'bcd'   (2번째부터 3글자, 1부터 시작)
LENGTH('abc')                 -- 3
REPLACE('abc','a','X')        -- 'Xbc'
TRIM('  ab  ')                -- 'ab'
NVL(성적, 0)                  -- NULL이면 0으로 (Oracle)
COALESCE(성적, 0)             -- 표준 SQL
ROUND(3.14159, 2)             -- 3.14
TO_CHAR(SYSDATE,'YYYY-MM-DD') -- 날짜 형식 변환
```

---

## 12. 시험에서 자주 틀리는 지점 체크

- [ ] `GRANT ~ TO` / `REVOKE ~ FROM` — 전치사 헷갈림
- [ ] `HAVING`을 `WHERE`로 잘못 씀 (집계 함수 조건)
- [ ] `NULL` 비교에 `=` 사용 (정답: `IS NULL`)
- [ ] `COUNT(*)` 와 `COUNT(컬럼)` 의 NULL 처리 차이
- [ ] `BETWEEN`은 양쪽 끝값 **포함**
- [ ] `DELETE`와 `DROP` 혼동
- [ ] `ALTER TABLE ~ DROP COLUMN` 에서 COLUMN 키워드 누락
- [ ] 문자열은 **작은따옴표** `'값'` (큰따옴표 아님)
- [ ] 세미콜론 `;` 누락
- [ ] `ORDER BY`의 기본값은 `ASC` (오름차순)